In [6]:
import pandas as pd  # Leer y manipular archivos Excel (.xlsx)

from docx import Document # Leer, modificar y guardar archivos Word (.docx)

from docx.shared import Inches # Sirve para especificar tamaños de las imagenes

import smtplib # Para conectarse a servidores de correo (Gmail)

from email.message import EmailMessage # Para crear mensajes de correo electrónico (asunto, cuerpo, destinatarios, adjuntos)

import os # Para interactuar con el sistema operativo (verificar archivos, crear carpetas, etc.)

# speech_recognition es opcional (solo si tienes archivo de audio)
try:
    import speech_recognition as sr
    SR_DISPONIBLE = True
except ImportError:
    SR_DISPONIBLE = False
    print("Nota: speech_recognition no está instalado. Se omitirá la transcripción de audio.")

print("Librerías importadas correctamente")

Librerías importadas correctamente


In [7]:
excel_file = 'Proyect.xlsx' # Archivo Excel con los datos de los empleados
word_file = 'Proyect.docx' # Archivo Word con la plantilla de la carta
image_file = 'Proyect.png' # Archivo de imagen para adjuntar a la carta
audio_file = 'Proyect.wav' # Archivo de audio para convertir a texto

email_sender = 'albionmy65@gmail.com' # Dirección de correo electrónico del remitente
email_password = 'yhxrzimzkccilmno' # Contraseña del correo electrónico del remitente
SMTP_server = 'smtp.gmail.com' # Servidor SMTP de Gmail
SMTP_PORT = 465 # Puerto SMTP de Gmail

print("Variables de configuración definidas correctamente")

Variables de configuración definidas correctamente


In [8]:
def reemplazar_texto_en_word(doc, diccionario):
    """Reemplaza placeholders en párrafos y tablas, preservando el formato original."""
    
    def reemplazar_en_parrafo(parrafo, diccionario):
        """Reemplaza texto en un párrafo operando sobre los runs para conservar formato."""
        texto_completo = parrafo.text
        encontrado = False
        for key in diccionario:
            if key in texto_completo:
                encontrado = True
                break
        if not encontrado:
            return
        # Reemplazar en el texto completo
        for key, value in diccionario.items():
            texto_completo = texto_completo.replace(key, value)
        # Limpiar los runs existentes y poner el texto nuevo en el primer run
        if parrafo.runs:
            # Guardar formato del primer run
            primer_run = parrafo.runs[0]
            primer_run.text = texto_completo
            # Limpiar el texto de los demas runs
            for run in parrafo.runs[1:]:
                run.text = ""
        else:
            parrafo.text = texto_completo
    
    # Reemplazar en párrafos normales
    for p in doc.paragraphs:
        reemplazar_en_parrafo(p, diccionario)
    
    # Reemplazar en tablas (si existen)
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for p in cell.paragraphs:
                    reemplazar_en_parrafo(p, diccionario)
    
    print("Texto reemplazado en el documento Word correctamente")

In [9]:
def automatizar_reporte_masivo():
    try:
        # Leer datos del Excel
        df = pd.read_excel(excel_file)
        print(f"Datos cargados: {len(df)} empleado(s) encontrado(s)")
        print(f"Columnas: {list(df.columns)}")
        
        # Transcribir audio (opcional)
        texto_audio = ""
        if SR_DISPONIBLE and os.path.exists(audio_file):
            recognizer = sr.Recognizer()
            with sr.AudioFile(audio_file) as source:
                audio_data = recognizer.record(source)
                texto_audio = recognizer.recognize_google(audio_data, language="es-ES")
            print(f"Audio transcrito: {texto_audio[:50]}...")
        
        enviados = 0
        errores = 0
        
        for idx, row in df.iterrows():
            nombre = str(row.get("Nombre", "Desconocido")).strip()
            apellido = str(row.get("Apellido", "")).strip()
            
            # FIX: La columna en el Excel se llama "Email", no "Correo"
            correo_destino = row.get("Email", "")
            
            if pd.isna(correo_destino) or str(correo_destino).strip() == "":
                print(f"  [{idx}] {nombre} {apellido} -> Sin correo, omitido")
                continue
            
            correo_destino = str(correo_destino).strip()
            print(f"  [{idx}] Procesando: {nombre} {apellido} ({correo_destino})")
            
            # Cargar plantilla Word fresca para cada empleado
            doc = Document(word_file)
            
            diccionario = {
                "[NOMBRE]": nombre,
                "[APELLIDO]": apellido,
                "[EDAD]": str(row.get("Edad", "")),
                "[TELEFONO]": str(row.get("Teléfono", row.get("Telefono", ""))),
                "[CARGO]": str(row.get("Cargo", "")),
                "[HONORARIOS]": str(row.get("Honorarios", "")),
            }
            
            reemplazar_texto_en_word(doc, diccionario)
            
            # Agregar imagen si existe
            if os.path.exists(image_file):
                doc.add_heading("Anexo Visual", level=1)
                doc.add_picture(image_file, width=Inches(4))
            
            # Agregar texto del audio si existe
            if texto_audio:
                doc.add_heading("Notas Transcritas", level=1)
                doc.add_paragraph(texto_audio)
            
            # Guardar reporte individual
            archivo_salida = f"Reporte_{nombre}_{apellido}.docx"
            doc.save(archivo_salida)
            print(f"    Reporte guardado: {archivo_salida}")
            
            # Enviar por correo
            try:
                msg = EmailMessage()
                msg["Subject"] = f"Tu Informe Corporativo Confidencial - {nombre}"
                msg["From"] = email_sender
                msg["To"] = correo_destino
                msg.set_content(f"Hola {nombre},\n\nAdjuntamos tu informe personalizado.\n\nSaludos.")
                
                with open(archivo_salida, "rb") as f:
                    msg.add_attachment(
                        f.read(),
                        maintype="application",
                        subtype="vnd.openxmlformats-officedocument.wordprocessingml.document",
                        filename=archivo_salida,
                    )
                
                with smtplib.SMTP_SSL(SMTP_server, SMTP_PORT) as server:
                    server.login(email_sender, email_password)
                    server.send_message(msg)
                
                print(f"    Correo enviado a: {correo_destino}")
                enviados += 1
            except Exception as email_error:
                print(f"    [Error al enviar correo]: {email_error}")
                errores += 1
        
        print(f"\n=== Resumen ===")
        print(f"Correos enviados: {enviados}")
        print(f"Errores: {errores}")
        print(f"Total procesados: {enviados + errores}")
    
    except Exception as e:
        import traceback
        print(f"[Error Crítico]: {e}")
        traceback.print_exc()

print("Estructura principal cargada.")

Estructura principal cargada.


In [10]:
automatizar_reporte_masivo()

Datos cargados: 1 empleado(s) encontrado(s)
Columnas: ['Nombre', 'Apellido', 'Edad', 'Email', 'Cargo', 'Honorarios']
  [0] Procesando: mine ghost (minecraftheghost69@gmail.com)
Texto reemplazado en el documento Word correctamente
    Reporte guardado: Reporte_mine_ghost.docx
    Correo enviado a: minecraftheghost69@gmail.com

=== Resumen ===
Correos enviados: 1
Errores: 0
Total procesados: 1
